In [1]:
import pandas as pd
import os
from nltk.tokenize import word_tokenize
import numpy as np
import csv
import matplotlib.pyplot as plt
import re

In [2]:
def clean_note(text):
    text = re.sub(r'^[\s"]+|[\s"]+$', '', text)   # strip edge quotes/spaces
    text = re.sub(r'\*', '', text)                 # remove asterisks
    text = re.sub(r'\s+', ' ', text)              # normalize whitespace
    return text.strip()

In [ ]:
def process_notes(input_note_file_path, generated_raw_notes_dir, output_dir):

    assert generated_raw_notes_dir != output_dir, "Input and output folders must be different!"

    prompt_df = pd.read_excel(input_note_file_path)

    sentence_lengths = []
    for file in os.listdir(generated_raw_notes_dir):
        if file.endswith('.csv'):
            temp_file_name = file.split('_')[1].replace('.csv', '')
            for note in pd.read_csv(f'{generated_raw_notes_dir}/{file}')['report'].to_list():
                sentence_lengths.append(len(word_tokenize(note)))
                
    l = np.array(sentence_lengths)

    q1 = np.quantile(l, 0.25)
    q3 = np.quantile(l, 0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    # store bounds
    upper_and_lower_bounds = {
        'lower': lower,
        'upper': upper
    }

    # Remove texts of x length above and below the Tukey fences in the list.
    for file in os.listdir(generated_raw_notes_dir):
        notes_for_output = []
        relevant_prompt = []
        file_ids = []
        raw_notes = []
        if file.endswith('.csv'):
            example_note_index = int(file.split('_')[0])
            # get prompt data index note
            needs = file.split('_')[1].replace('.csv', '')
            prompt = prompt_df[prompt_df['Needs'] == needs]['Note'].tolist()[example_note_index]
            notes = pd.read_csv(f'{generated_raw_notes_dir}/{file}')['report'].to_list()
            raw_notes.extend(notes)
            for note in notes:
                # raw_segments = re.split(r'\s*"+(?:\s*"+)+\s*', note)
                raw_segments = re.split(r'\s*(?:"{2,}\s*){1,}"+\s*', note)
                for segment in raw_segments:
                    segment = clean_note(segment)

                    if not segment:
                        continue

                    temp_length = len(word_tokenize(segment))
                    # Filter based on lower bound vs 5 words max and upper bound. 
                    if max(upper_and_lower_bounds['lower'], 5) <= temp_length <= upper_and_lower_bounds['upper']:
                        # remove any note with accidental reference to the topic model keywords
                        if 'other topic keywords:' not in segment.lower():
                            # Remove accidental reference to the input prompt.
                            if prompt.lower() not in segment.lower():
                                relevant_prompt.append(prompt)
                                notes_for_output.append(segment)
                                file_ids.append(file.replace('.csv', ''))
                            else:
                                # print(f'{dataset} DIRECT PROMPT REFERENCE {prompt} - {note}')
                                pass
                        else:
                            # print(f'{dataset} DIRECT KEYWORDS REFERENCE {prompt} - {note}')
                            pass
                    else:
                        # print(f"{dataset} TOO LONG {note} is {temp_length} tokens long, which falls outside the range of {upper_and_lower_bounds[dataset][temp_topic_model]['lower']} - {upper_and_lower_bounds[dataset][temp_topic_model]['upper']}")
                        pass
        os.makedirs(f'./{output_dir}/', exist_ok=True)
        output_df = pd.DataFrame({
            'file_id' : file_ids,
            'example_text_in_prompt' : relevant_prompt,
            'report' : notes_for_output,
        })
        output_df = output_df[output_df['report'].notna() & (output_df['report'] != "")]
        output_df = output_df.drop_duplicates(subset=['report'])
        output_df.to_csv(f'./{output_dir}/{file}', index=False, quoting=csv.QUOTE_ALL, encoding='utf-8', lineterminator='\n')
        print(f"{file}: {len(raw_notes)} vs {len(output_df)}")

In [4]:
process_notes(f'./syntheticNotesLocal/realNoteGeneration/real_notes.xlsx', f'./syntheticNotesLocal/realNoteGeneration/realNoteSyntheticNotes', f'./syntheticNotesLocal/realNoteGeneration/processedRealNoteSyntheticNotes')
process_notes(f'./syntheticNotesLocal/fakeNoteGeneration/fake_notes.xlsx', f'./syntheticNotesLocal/fakeNoteGeneration/fakeNoteSyntheticNotes', f'./syntheticNotesLocal/fakeNoteGeneration/processedFakeNoteSyntheticNotes')

0_met.csv: 270 vs 259
0_unmet.csv: 258 vs 244
1_met.csv: 243 vs 238
1_unmet.csv: 329 vs 322
2_met.csv: 242 vs 232
2_unmet.csv: 225 vs 224
3_met.csv: 262 vs 253
3_unmet.csv: 320 vs 324
4_met.csv: 269 vs 265
4_unmet.csv: 277 vs 271
0_met.csv: 224 vs 216
0_unmet.csv: 310 vs 300
1_met.csv: 236 vs 228
1_unmet.csv: 219 vs 210
2_met.csv: 239 vs 232
2_unmet.csv: 285 vs 278
3_met.csv: 291 vs 272
3_unmet.csv: 247 vs 238
4_met.csv: 183 vs 170
4_unmet.csv: 256 vs 241
